In [1]:
from typing import Annotated
from typing_extensions import TypedDict

from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_ollama import ChatOllama
from langchain_core.tools import tool

from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

#-------------------------------------------Tools-----------------------------------------\
from tools_local.speak import speak_text
from tools_local.show_notification import show_notification
from tools_local.open_taskmgr import open_task_manager
from tools_local.open_folder import open_folder
from tools_local.take_shot import take_screenshot

from tools_local.window_interact import windows_tool
from tools_local.weather import get_weather
from tools_local.save_python_file import save_python_file
from tools_local.open_designer import open_qt_designer
from tools_local.open_web_page import open_web_page

from langchain_core.tools import tool
import os


@tool
def clear_terminal() -> str:
    """
    Clear the terminal or command prompt screen.

    Use this tool whenever the user asks to:
    - clear the terminal
    - clear the console
    - clear the command prompt
    - clear the CMD screen
    - clean the terminal
    - reset the console display
    """

    try:

        if os.name == "nt":
            os.system("cls")
        else:
            os.system("clear")

        print("Screen cleared by Abhishek Verma")

        return "Terminal screen cleared successfully."

    except Exception as e:

        return f"Failed to clear terminal.\n{e}"
#--------------------------------------------End-----------------------------------------/

# Using Tool supported chatmodel LLm from ollama Api
llm = ChatOllama(model="gemma4:31b-cloud",temperature=0,) #gemma4:31b-cloud,qwen3.5:4b
tools = [
    open_task_manager,
    show_notification,
    speak_text,
    clear_terminal,
    open_folder,
    take_screenshot,
    windows_tool,
    get_weather,
    save_python_file,
    open_qt_designer,
    open_web_page,
    
    ] # for Tool Registration for Agent

llm = llm.bind_tools(tools) # Agent tool connection


class Memory(TypedDict): 
    messages: Annotated[list, add_messages]



def chatbot(state: Memory):
    # system message is the prompt to instruct llm to tell the model how to use tools and follow rules
    SYSTEM_MESSAGE = """
        You are Jarvis, a desktop AI assistant.

        Rules:

        - Always use tools whenever a suitable tool exists.
        - Never make up tool results.
        - Wait for a tool to finish before responding.
        - Use the minimum number of tools needed.
        - If a tool fails, explain the error.
        - Keep responses short and direct.
        - When asked to open Windows applications, use the appropriate Windows tool.
        - When asked to speak, use the Speak tool.
        - When asked to show a notification, use the Notification tool.
        - you can use the Speak tool at the end to tell the user if you want them to hear a response.
    """
    messages = [
        SystemMessage(content=SYSTEM_MESSAGE),
        *state["messages"],
    ]

    response = llm.invoke(messages)

    return {
        "messages": [response]
    }


graph = StateGraph(Memory)
graph.add_node("chatbot", chatbot)
graph.add_node("tools",ToolNode(tools))
graph.add_edge(START,"chatbot")
graph.add_conditional_edges("chatbot",tools_condition,)
graph.add_edge("tools","chatbot")
app = graph.compile()


print("Jarvis Started")
print("Type 'close' to quit.\n")

while True:
    user_input = input("You : ")

    if user_input.lower() == "close":
        break

    print("\nJarvis : ", end="", flush=True)
    send_to_llm = app.stream({"messages": [HumanMessage(user_input)]},stream_mode="messages",)
    
    for message, metadata in send_to_llm:
        if hasattr(message, "content") and message.content:
            print(message.content, end="", flush=True)

    print("\n")

c:\Projects\Models\agentic\LangFlow\langflow_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Projects\Models\agentic\LangFlow\langflow_env\Lib\site-packages\torch\cuda\__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


Jarvis Started
Type 'close' to quit.


Jarvis : Hello! How can I assist you today?


Jarvis : Task Manager opened successfully.I have opened the Task Manager for you.

